In [ ]:
%pip install langchain
%pip install openai
%pip install chromadb
%pip install pypdf
%pip install streamlit
%pip install langchain-openai
%pip install python-docx
%pip install langchain-text-splitters
%pip install sentence-transformers
%pip install langchain-huggingface
%pip install python-dotenv
%pip install google-generativeai

In [6]:
from pypdf import PdfReader

reader = PdfReader("resume.pdf")

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print(text)

Vikaskumar Dubey 
8097440042 dubeyvikasku@gmail.com linkedin.com/in/vikashere Mumbai, India 
 
SUMMARY 
Full-stack software engineer with 2+ years of experience in backend API development, Python application 
engineering, and system automation. Built a production-style RAG-based LLM application integrating OpenAI 
APIs, semantic search, and a conversational interface — demonstrating hands-on AI integration capability. Strong 
CS fundamentals (350+ LeetCode problems, Smart India Hackathon finalist) with a risk -and-control mindset 
shaped by security-critical enterprise work at Capgemini. Eager to apply Python-first engineering and LLM 
integration skills to model risk governance tooling at JPMorgan Chase. 
 
TECHNICAL SKILLS 
Languages Python (primary), Java, SQL, JavaScript, C++, Bash 
Frameworks Django, Spring Boot, Spring REST, FastAPI, React, Bootstrap 
LLM / AI OpenAI API, LangChain, RAG pipelines, ChromaDB (vector search), prompt engineering 
Data pandas, NumPy, MySQL, SQL Server

In [28]:

import re

sections = re.split(
    r'\n\s*(?=[A-Z][A-Z\s]{3,}\n)',
    text
)

chunks = [s.strip() for s in sections if s.strip()]

print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"\n===== CHUNK {i} =====")
    print(chunk[:500])


Total chunks: 6

===== CHUNK 0 =====
Vikaskumar Dubey 
8097440042 dubeyvikasku@gmail.com linkedin.com/in/vikashere Mumbai, India

===== CHUNK 1 =====
SUMMARY 
Full-stack software engineer with 2+ years of experience in backend API development, Python application 
engineering, and system automation. Built a production-style RAG-based LLM application integrating OpenAI 
APIs, semantic search, and a conversational interface — demonstrating hands-on AI integration capability. Strong 
CS fundamentals (350+ LeetCode problems, Smart India Hackathon finalist) with a risk -and-control mindset 
shaped by security-critical enterprise work at Capgemini. 

===== CHUNK 2 =====
TECHNICAL SKILLS 
Languages Python (primary), Java, SQL, JavaScript, C++, Bash 
Frameworks Django, Spring Boot, Spring REST, FastAPI, React, Bootstrap 
LLM / AI OpenAI API, LangChain, RAG pipelines, ChromaDB (vector search), prompt engineering 
Data pandas, NumPy, MySQL, SQL Server, SQLite, REST API design 
Architecture OOP, d

In [29]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2323.83it/s]


In [32]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="resume"
)

for i, chunk in enumerate(chunks):
    vector = embedding_model.encode(chunk).tolist()

    collection.add(
        ids=[str(i)],
        documents=[chunk],
        embeddings=[vector]
    )

In [33]:
from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

response = model.generate_content("Hello")
print(response.text)

Hello! How can I help you today?


In [35]:
def retrieve_context(question):

    query_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    return "\n".join(
        results["documents"][0]
    )

In [36]:
def ask_resume(question):

    context = retrieve_context(question)

    prompt = f"""
You are Vikas's AI Resume Assistant.

Rules:
1. Answer only from the provided resume context.
2. Do not make up information.
3. If information is not present, say:
   "I could not find that information in the resume."

Resume Context:
{context}

Question:
{question}
"""

    response = model.generate_content(prompt)

    return response.text


In [39]:
print(
    ask_resume(
        "What programming languages does Vikas know?"
    ),
    ask_resume("Tell me about Vikas's work experience.")
)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 4.865743205s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 4
}
]

In [38]:

while True:

    question = input("You: ")

    if question.lower() == "exit":
        break

    answer = ask_resume(question)

    print("\nBot:", answer)



Bot: **Associate Software Engineer** · Capgemini Technology Services (Ericsson) May 2024 – Present
*   **Authentication Federation Gateway (AFG) — Enterprise Security Platform**
    *   Developed and maintained backend API services in Java (Spring Boot) and C++ for the AFG platform, handling authentication federation across distributed enterprise nodes.
    *   Implemented and optimized OAuth 2.0 and JWT authentication protocols, improving token validation reliability and contributing to security compliance across multi-vendor environments.
    *   Built backend data processing modules responsible for ingesting and handling authentication events, with strict attention to data integrity, audit trails, and control mechanisms.
    *   Collaborated with cross-functional teams (product, QA, security architects) to translate business requirements into backend deliverables, participating in design discussions, code reviews, and incident response.
    *   Worked in an agile environment with f

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 11.788611307s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 11
}
]

In [25]:
import streamlit as st

st.title("Vikas Resume Assistant")

question = st.text_input(
    "Ask a question about my resume"
)

if question:
    answer = ask_resume(question)
    st.write(answer)


2026-07-03 13:43:22.887 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 13:43:22.889 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 13:43:22.890 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 13:43:22.891 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 13:43:22.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 13:43:22.911 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 13:43:22.913 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 13:43:22.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [31]:
client.delete_collection("resume")